# 第9章 映像から動きを逆算する (Inverse Dynamics)

第8章の逆です。**動画を渡して、その動画のあいだにカメラ (車) がどう動いたか**を、行動の数値で答えさせます。逆動力学 (inverse dynamics) と呼びます。

- ドライブレコーダーの映像から、車の走った道筋を復元する (GPS がなくても)
- ロボットの作業動画から、手先の動きを取り出す (人のお手本動画をロボットの教師データにする研究につながる)
- 第8章で作った予測動画が、指示した行動どおりになっているかを確かめる

この章で試すこと:

1. お手本の車載映像 2 本から、車の道筋を復元する (モデルカードの答えと比べる)
2. 往復チェック: 行動 → 予測動画 (第8章) → 行動 (この章) で、指示と復元が一致するか
3. 自分のドライブレコーダーの映像で試す

設定は公式の推奨どおりです: 61 コマ (10fps で 6 秒) の動画から 60 ステップの行動、480p、`num_inference_steps=30`、`guidance_scale=1.0`、`flow_shift=10.0`。

## 準備

**ランタイム**: 「ランタイム → ランタイムのタイプを変更」で **L4** か **A100** の GPU を選んでください (T4 は対象外)。
A100 (40GB 以上) ではモデル全体を GPU に載せ、L4 (24GB) では使う部品だけを GPU に載せる設定 (CPU オフロード) に自動で切り替えます。

**安全フィルタ**: 生成には NVIDIA の安全フィルタ (`cosmos_guardrail`) を使います。使うには Hugging Face で
[nvidia/Cosmos-1.0-Guardrail](https://huggingface.co/nvidia/Cosmos-1.0-Guardrail) の利用申請をして承認を受け、
Colab の左の鍵のマーク (シークレット) に `HF_TOKEN` を登録しておいてください。承認を待たずに試すときは、下の `GUARDRAIL` の印を外します。

In [ ]:
# @title GPU を確かめる
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
if DEVICE == "cuda":
    GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(torch.cuda.get_device_name(0), f"{GPU_GB:.0f} GB")
    assert torch.cuda.get_device_capability()[0] >= 8, "Ampere 以降の GPU (L4 / A100 など) を選んでください。T4 では動きません。"
else:
    GPU_GB = 0
    print(f"CUDA の GPU が見つかりません ({DEVICE} で動かします。とても遅くなります)")

In [ ]:
# @title インストール (数分)
# Cosmos3OmniPipeline は Diffusers の main ブランチに入っている
!pip install -q -U "transformers>=5.18" accelerate av imageio imageio-ffmpeg huggingface_hub
!pip install -q "diffusers @ git+https://github.com/huggingface/diffusers.git" cosmos_guardrail

In [ ]:
# @title 安全フィルタの設定とサンプル素材を取ってくる関数
import os
import urllib.request
from pathlib import Path

GUARDRAIL = True  # @param {type:"boolean"}
try:  # Colab のシークレットに HF_TOKEN があれば使う
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

ASSETS = Path(os.environ.get("COSMOS_ASSETS", "assets"))
ASSETS.mkdir(parents=True, exist_ok=True)
FW = "https://media.githubusercontent.com/media/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
HF = "https://huggingface.co/nvidia/Cosmos3-Edge/resolve/main/assets"


def fetch(url):
    """URL の素材を assets/ に落として、そのパスを返す (2 回目からは落とさない)"""
    path = ASSETS / url.rsplit("/", 1)[-1]
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return str(path)

In [ ]:
# @title 生成モデル (Generator) を読み込む関数
import gc
import json

from diffusers import Cosmos3OmniPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video
from IPython.display import Image as IPImage, Video, display
from PIL import Image, ImageOps


def load_image(path, max_side=10_000):
    """画像を RGB で読む (max_side より大きければ縮める)"""
    img = path if isinstance(path, Image.Image) else ImageOps.exif_transpose(Image.open(path))
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img


def load_pipe(flow_shift):
    """Cosmos3OmniPipeline を読み込む (2 回目からは読み込み済みのものを使う)。
    flow_shift は公式の推奨値: 画像→動画は 12.0、行動 (action) を使う生成は 10.0"""
    global pipe
    if "pipe" not in globals():
        pipe = Cosmos3OmniPipeline.from_pretrained(
            "nvidia/Cosmos3-Edge", torch_dtype=torch.bfloat16, enable_safety_checker=GUARDRAIL)
        if DEVICE == "cuda" and GPU_GB < 40:
            pipe.enable_model_cpu_offload()  # L4 など: 使う部品だけ GPU に載せる
        else:
            pipe.to(DEVICE)
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=flow_shift, use_karras_sigmas=False)
    return pipe


def free_gpu():
    """読み込んだモデルを GPU から降ろす"""
    for name in ("pipe", "model", "processor"):
        globals().pop(name, None)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def show_video(path, width=640):
    display(Video(path, embed=True, width=width))

In [ ]:
# @title 行動 (action) を作る・描く関数
import numpy as np
import matplotlib.pyplot as plt
from diffusers import CosmosActionCondition

RAW = "https://raw.githubusercontent.com/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
AV_PROMPT = "You are an autonomous vehicle planning system."
# AV の行動は「1 コマ前のカメラから見た、次のコマのカメラの位置と向き」を 9 個の数で表す。
#   [x, y, z, r1..r6]  x: 右, y: 下, z: 前 (OpenCV のカメラ座標)。r1..r6: 回転行列の 1 列目と 2 列目 (rot6d)
# 移動量はメートルを 1.35 倍した値 (公式の可視化コードと同じ換算)
AV_SCALE = 1.35


def rot_y(theta):
    """カメラの下向きの軸 (y) まわりの回転。theta が負だと左を向く"""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def from_rot6d(r):
    a = np.asarray(r[:3], float)
    b = np.asarray(r[3:6], float)
    c0 = a / np.linalg.norm(a)
    c1 = b - c0 * (c0 @ b)
    c1 /= np.linalg.norm(c1)
    return np.stack([c0, c1, np.cross(c0, c1)], axis=1)


def make_av_actions(speed_kmh=30.0, left_deg_per_s=0.0, accel=0.0, seconds=6.0, fps=10):
    """速さ (km/h)・左へ曲がる速さ (度/秒、右は負)・加速度 (m/s^2、減速は負) から AV の行動 (T, 9) を作る。
    left_deg_per_s には「時刻 t (秒) → 度/秒」の関数も渡せる (車線変更などに)"""
    rows, dt = [], 1 / fps
    for i in range(int(round(seconds * fps))):
        t = i * dt
        v = max(speed_kmh / 3.6 + accel * t, 0.0)
        d = v * dt
        yaw = left_deg_per_s(t) if callable(left_deg_per_s) else left_deg_per_s
        th = -np.radians(yaw) * dt if d > 0 else 0.0
        move = np.array([np.sin(th / 2) * d, 0.0, np.cos(th / 2) * d]) * AV_SCALE
        R = rot_y(th)
        rows.append(np.concatenate([move, R[:, 0], R[:, 1]]))
    return np.array(rows, np.float32)


def to_path(actions, scale=AV_SCALE):
    """行動 (T, 9) をつなげて、上から見た道筋 (T+1, 3) にする。x: 右, z: 前 (メートル)"""
    R, p, pts = np.eye(3), np.zeros(3), [np.zeros(3)]
    for a in np.asarray(actions, float):
        p = p + R @ (a[:3] / scale)
        R = R @ from_rot6d(a[3:9])
        pts.append(p.copy())
    return np.array(pts)


def plot_paths(paths, title="top-down view", fps=10):
    """{名前: 行動 (T, 9)} を上から見た図に描く。点は 1 秒ごと"""
    fig, ax = plt.subplots(figsize=(5, 5))
    for i, (name, actions) in enumerate(paths.items()):
        p = to_path(actions)
        ax.plot(p[:, 0], p[:, 2], lw=2, label=name)
        ax.scatter(p[::fps, 0], p[::fps, 2], s=12)
    ax.scatter([0], [0], c="k", marker="^", s=80, zorder=5)
    ax.set_xlabel("x: right (m)")
    ax.set_ylabel("z: forward (m)")
    ax.set_aspect("equal", adjustable="datalim")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    ax.set_title(title)
    plt.show()


def tile_videos(videos, labels, out, fps, cols=2):
    """同じ長さの動画 (PIL 画像のリスト) を並べて 1 本にする"""
    from PIL import ImageDraw, ImageFont
    n = min(len(v) for v in videos)
    w, h = videos[0][0].size
    rows = (len(videos) + cols - 1) // cols
    font = ImageFont.load_default(size=max(16, w // 25))
    frames = []
    for t in range(n):
        canvas = Image.new("RGB", (w * cols, h * rows))
        for k, (v, label) in enumerate(zip(videos, labels)):
            tile = v[t].convert("RGB").resize((w, h))
            ImageDraw.Draw(tile).text((10, 8), label, fill="white", font=font, stroke_width=3, stroke_fill="black")
            canvas.paste(tile, ((k % cols) * w, (k // cols) * h))
        frames.append(canvas)
    export_to_video(frames, out, fps=fps, macro_block_size=1)
    return out

In [ ]:
# @title 動画を 10fps・61 コマ・832x480 に揃える関数
import av
from PIL import ImageOps


def read_clip(path, fps=10, n=61, size=(832, 480), start=0.0):
    """動画を fps に間引いて start 秒から n コマ取り出し、size に中央で切り抜く"""
    frames, next_t = [], start
    with av.open(path) as c:
        for f in c.decode(video=0):
            if f.time is None or f.time + 1e-6 < next_t:
                continue
            frames.append(ImageOps.fit(f.to_image().convert("RGB"), size, Image.BICUBIC))
            next_t += 1 / fps
            if len(frames) == n:
                break
    if len(frames) < n:
        raise ValueError(f"{path} は {start} 秒から {n / fps:.1f} 秒ぶんの長さがありません ({len(frames)} コマ)")
    return frames


def estimate_motion(frames, seed=0):
    """61 コマの動画から、60 ステップの AV の行動 (60, 9) を返す"""
    pipe = load_pipe(flow_shift=10.0)
    result = pipe(
        prompt=AV_PROMPT,
        action=CosmosActionCondition(
            mode="inverse_dynamics",
            chunk_size=len(frames) - 1,
            domain_name="av",
            resolution_tier=480,
            video=frames,
            view_point="ego_view",
        ),
        fps=10,
        num_inference_steps=30,
        guidance_scale=1.0,
        use_system_prompt=False,
        generator=torch.Generator(device=DEVICE).manual_seed(seed),
    )
    return result.action[0].float().cpu().numpy()

## 1. お手本の車載映像から、車の道筋を復元する

モデルカードには、同じ動画を Cosmos3-Edge に入れたときの答え (行動の JSON) も置かれています。自分で動かした結果と重ねて描きます。
完全には一致しませんが (生成は毎回少し違う)、道筋の形が近ければ正しく動いています。

In [ ]:
for k in (0, 1):
    video = fetch(f"{HF}/example_action_id_av_{k}_input.mp4")
    show_video(video, 480)
    frames = read_clip(video)
    mine = estimate_motion(frames)
    ref = np.array(json.loads(Path(fetch(f"{HF}/edge_action_id_av_{k}_output.json")).read_text())["data"], np.float32)
    print(f"clip {k}: 復元した行動 {mine.shape}, 6 秒で約 {np.linalg.norm(to_path(mine)[-1]):.0f} m 進んだ")
    plot_paths({"this notebook": mine, "model card": ref.reshape(-1, 9)}, f"clip {k}: recovered ego path")

## 2. 往復チェック: 行動 → 予測動画 → 行動

第8章と同じように、決めた行動 (左折) で予測動画を作り、その動画からこの章の方法で行動を当て直します。
指示した道筋と、当て直した道筋が近ければ、**予測動画が行動どおりに描けている**ことの確かめになります。
世界モデルの出来を数字で測る、簡単で実用的な方法です。

In [ ]:
start = load_image(fetch(f"{FW}/generator/action/assets/images/av_0.jpg"))
command = make_av_actions(speed_kmh=25, left_deg_per_s=12)

pipe = load_pipe(flow_shift=10.0)
result = pipe(
    prompt=AV_PROMPT,
    action=CosmosActionCondition(mode="forward_dynamics", chunk_size=len(command), domain_name="av", resolution_tier=480,
                                 raw_actions=torch.as_tensor(command), image=start, view_point="ego_view"),
    fps=10, num_inference_steps=30, guidance_scale=1.0, use_system_prompt=False,
    generator=torch.Generator(device=DEVICE).manual_seed(0),
)
export_to_video(result.video, "ch09_predicted.mp4", fps=10, macro_block_size=1)
show_video("ch09_predicted.mp4", 480)

recovered = estimate_motion(result.video)
plot_paths({"command (input)": command, "recovered from video": recovered}, "round trip: action -> video -> action")
err = np.linalg.norm(to_path(command)[:, [0, 2]] - to_path(recovered)[:, [0, 2]], axis=1)
print(f"道筋のずれ: 平均 {err.mean():.1f} m、6 秒後 {err[-1]:.1f} m (6 秒で {np.linalg.norm(to_path(command)[-1]):.0f} m 進む指示)")

## 3. 自分のドライブレコーダーの映像で試す

6 秒以上の動画を入れてください。`start_sec` 秒から 6 秒ぶんを 10fps に間引いて使います。

In [ ]:
# @title 自分の動画で試す (colab-only)
from google.colab import files

start_sec = 0.0  # @param {type:"number"}
crop_bottom = 0.0  # @param {type:"slider", min:0, max:0.3, step:0.01}

up = files.upload()
path = next(iter(up))
frames = read_clip(path, start=start_sec, size=(832, round(480 / (1 - crop_bottom))))
frames = [f.crop((0, 0, 832, 480)) for f in frames]
export_to_video(frames, "ch09_my_clip.mp4", fps=10, macro_block_size=1)
show_video("ch09_my_clip.mp4", 480)
mine = estimate_motion(frames)
plot_paths({"my drive": mine}, "recovered ego path")

## うまくいかないとき

- **道筋の長さ (速さ) がずれる**: 1 台のカメラの映像だけでは、距離の絶対的な大きさは原理的にあいまいです。向き (曲がったかどうか) のほうが信頼できます
- **止まっているのに進んだことになる**: 周りの車が動いていると取り違えることがあります。seed を変えて何回か当て、平均を取ると安定します
- **ロボットの動画で使いたい**: ロボットの種類ごとに行動の形が違います (UMI・DROID・Bridge など公式に対応したものだけ)。公式の `run_id_with_diffusers.ipynb` に例があります